In [1]:
# from comet_ml import Experiment
# from comet_ml.integration.pytorch import log_model

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import torch.nn.functional as F
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import pickle as pkl

import matplotlib.pyplot as plt
import numpy as np
import io, os
from tqdm import tqdm

from PIL import Image
from torchvision import models 
from torchvision.models import resnet18

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import KBinsDiscretizer
import statsmodels.api as sm

import lightning.pytorch as pl
import torch
import torch.nn as nn
import torchmetrics
import torchvision
from torchvision import models

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, multilabel_confusion_matrix

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
old_data_dir = '/media/ssd1/huong/PCR-huong/data/'
new_data_dir = os.path.join(old_data_dir,'new_data')

In [17]:
class SequenceGeneDataModule(pl.LightningDataModule):
    """
        Pytorch Lightning DataModule for Image+Sequence dataset. This will download the dataset, prepare data loaders and apply
        data augmentation.
    """
    def __init__(self, curve_dict_path, target_df_path, norm_mean=None, norm_std=None, batch_size=32, shuffle=True, num_workers=4, igi_call=False):
        super().__init__()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.igi_call = igi_call
        # self.img_directory = img_directory
        self.norm_mean = norm_mean
        self.norm_std = norm_std

        print("WE ARE USING THE IMAGE SEQUENCE GENE DATASET")

        with open(curve_dict_path, 'rb') as file:
            self.curve_dict = pkl.load(file)
        
        self.target_df = pd.read_csv(target_df_path)

        self.target_df['igi_fp'] = (self.target_df['Igi_call_quant'] > self.target_df['groundtruth_target']).astype(int)
        self.target_df['igi_fn'] = (self.target_df['Igi_call_quant'] < self.target_df['groundtruth_target']).astype(int)

        self.target_df_train = self.target_df[self.target_df['split']=='train']
        self.curve_dict_train = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_train['curve_idx'].values}
        
        self.target_df_val = self.target_df[self.target_df['split']=='val']
        self.curve_dict_val = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_val['curve_idx'].values}

        self.target_df_test = self.target_df[self.target_df['split']=='test']
        self.curve_dict_test = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_test['curve_idx'].values}

        if self.norm_mean is None:
            mean_list = []
            std_list = []
            
            for key, curve in tqdm(self.curve_dict_train.items()):
                mean_curve = np.array(curve).mean().item()
                std_curve = np.array(curve).std().item()

                mean_list.append(mean_curve)
                std_list.append(std_curve)

            self.norm_mean = np.array(mean_list).mean().item()
            self.norm_std = np.array(std_list).mean().item()
            
        self.setup()

    def prepare_data(self):
        return

    def setup(self, stage=None):
        self.train = SequenceGeneDataset(self.curve_dict_train, self.target_df_train, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.val = SequenceGeneDataset(self.curve_dict_val, self.target_df_val, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.test = SequenceGeneDataset(self.curve_dict_test, self.target_df_test, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True, num_workers = self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)


class SequenceGeneDataset(Dataset):
    def __init__(self, curve_dict, target_df, sequence_len=40, igi_call=False,
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        
        target_ls = ['target_' + t for t in ['S gene','N gene','E gene','RnaseP','MS2','ORF1ab']]

        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        
        if self.one_hot.shape[1] != 6:
            for target in target_ls:
                if target not in self.one_hot.columns:
                    self.one_hot[target] = False
        self.one_hot = self.one_hot[['target_E gene', 'target_MS2','target_N gene',
                                                  'target_ORF1ab','target_RnaseP','target_S gene']]
        self.target_df = pd.concat([self.target_df, 
                                    self.one_hot], axis=1)

        # self.img_directory = img_directory
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std
        self.igi_call = igi_call

        # Image transformations: Resize and Normalize
        # self.img_transforms = transforms.Compose([
        #     transforms.Lambda(lambda image: image.convert('RGB')),
        #     transforms.Resize((224, 224)),  # Resizing to a consistent size
        #     transforms.ToTensor(),  # Convert PIL image to tensor
        #     transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
        #     ])
   
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        # curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        # curve_img = Image.open(curve_img_path)
        # curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        gene_type = torch.tensor(row[self.one_hot.columns].values, dtype=torch.float32)

        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.float)

        if self.igi_call:
            igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.float)
            igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.float)
            target = torch.stack([target, igi_fp, igi_fn], dim=0)

        return sequence_normalized.unsqueeze(1), gene_type.squeeze(1), target, curve_idx


In [18]:
class Classifier(pl.LightningModule):
    def __init__(self, num_classes=2, init_lr=1e-4):
        super().__init__()
        self.init_lr = init_lr
        self.num_classes = num_classes

        # Define loss fn for classifier
        self.loss = nn.BCELoss()

        self.accuracy = torchmetrics.Accuracy(task="binary" if self.num_classes == 2 else "multiclass", num_classes=self.num_classes)
        self.auc = torchmetrics.AUROC(task="binary" if self.num_classes == 2 else "multiclass", num_classes=self.num_classes)

        self.training_outputs = []
        self.validation_outputs = []

    def get_xy(self, batch):
        x, y = batch[0], batch[1]
        return x, y

    def training_step(self, batch, batch_idx):
        x, y = self.get_xy(batch)

        ## TODO: get predictions from your model and store them as y_hat
        y_hat = self.forward(*x)
        #y_hat = self.forward(x)
        loss = sum(self.loss(y_hat[:,i],y[:,i]) for i in range(3))

        self.log('train_loss', loss, prog_bar=True, sync_dist=True)

        ## Store the predictions and labels for use at the end of the epoch
        self.training_outputs.append({
            "y_hat": y_hat,
            "y": y
        })
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = self.get_xy(batch)

        y_hat = self.forward(*x)
        #y_hat = self.forward(x)
        loss = sum(self.loss(y_hat[:,i],y[:,i]) for i in range(3))

        self.log('val_loss', loss, prog_bar=True, sync_dist=True)

        self.validation_outputs.append({
            "y_hat": y_hat,
            "y": y
        })
        return loss

    def test_step(self, batch, batch_idx):
        x, y = self.get_xy(batch)

        y_hat = self.forward(*x)

        #loss = self.loss(y_hat,y)
        loss = sum(self.loss(y_hat[:,i],y[:,i]) for i in range(3))

        self.log('test_loss', loss, sync_dist=True, prog_bar=True)
        self.log('test_acc', self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)

        self.test_outputs.append({
            "y_hat": y_hat,
            "y": y
        })
        return loss
    
    def on_train_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.training_outputs])
        y = torch.cat([o["y"] for o in self.training_outputs])
        
        self.log("train_auc", self.auc(y_hat, y), sync_dist=True, prog_bar=True)
        self.log("train_acc", self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)
        self.training_outputs = []

    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])
        
        self.log("val_auc", self.auc(y_hat, y), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)
        #self.validation_outputs = []
        
        # save to process later for evaluation
        torch.save(y_hat, 'y_hat_val_image.pt')
        torch.save(y, 'y_val_true.pt')
    
    def on_test_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.test_outputs])
        y = torch.cat([o["y"] for o in self.test_outputs])

        if self.num_classes == 2:
            probs = F.softmax(y_hat, dim=-1)[:,-1]
        else:
            probs = F.softmax(y_hat, dim=-1)

        self.log("test_auc", self.auc(probs, y.view(-1)), sync_dist=True, prog_bar=True)

        self.log("val_auc", self.auc(y_hat, y), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)
        self.test_outputs = []

        # save to process later for evaluation
        torch.save(y_hat, 'y_hat_test_image.pt')
        torch.save(y, 'y_test_true.pt')

    def configure_optimizers(self):
        ## TODO: Define your optimizer and learning rate scheduler here (hint: Adam is a good default)

        optimizer = torch.optim.Adam(self.parameters(), lr=self.init_lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

        # return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'monitor':'val_loss'}}
        return optimizer


class SeqModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = seq_latent
        # fusion = torch.cat((seq_latent), dim=1)
        # fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), seq_latent_delta), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []


In [30]:

test_datamodule = SequenceGeneDataModule(curve_dict_path=os.path.join(new_data_dir, 'new_groundtruth_curve_dict.pkl'),
                             target_df_path=os.path.join(new_data_dir, 'new_groundtruth_target_data.csv'),
                             num_workers=32)
testloader = test_datamodule.test_dataloader()

# retest_datamodule = SequenceGeneDataModule(curve_dict_path=os.path.join(new_data_dir, 'new_retest_curve_dict.pkl'),
#                              target_df_path=os.path.join(new_data_dir, 'new_retest_df_target_data.csv'),
#                              norm_mean=test_datamodule.norm_mean,
#                              norm_std=test_datamodule.norm_std,
#                              num_workers=32)
# retestloader = retest_datamodule.test_dataloader()

# retested_sample_datamodule = SequenceGeneDataModule(curve_dict_path=os.path.join(new_data_dir, 'new_retested_curve_dict.pkl'),
#                              target_df_path=os.path.join(new_data_dir, 'new_retested_df_target_data.csv'),
#                              norm_mean=test_datamodule.norm_mean,
#                              norm_std=test_datamodule.norm_std,
#                              num_workers=32)
# retestedsampleloader = retested_sample_datamodule.test_dataloader()

# with open(os.path.join(old_data_dir, 'chip60_curve_dict.pkl'),'rb') as f:
#     mean_ls = []
#     std_ls = []
#     curve_dict = pkl.load(f)
#     for v in curve_dict.values():
#         mean_ls.append(np.array(v).mean().item())
#         std_ls.append(np.array(v).std().item())
#     chip_norm_mean = np.array(mean_ls).mean().item()
#     chip_norm_std = np.array(std_ls).mean().item()
        
# chip_datamodule = SequenceGeneDataModule(curve_dict_path=os.path.join(old_data_dir, 'chip60_curve_dict.pkl'),
#                              target_df_path=os.path.join(old_data_dir, 'chip60_target_data.csv'),
#                              norm_mean=chip_norm_mean,
#                              norm_std=chip_norm_std,
#                              num_workers=32)
# chip60loader = chip_datamodule.test_dataloader()

# with open(os.path.join(old_data_dir, 'karlen_curve_dict.pkl'),'rb') as f:
#     mean_ls = []
#     std_ls = []
#     curve_dict = pkl.load(f)
#     for v in curve_dict.values():
#         mean_ls.append(np.array(v).mean().item())
#         std_ls.append(np.array(v).std().item())
#     karlen_norm_mean = np.array(mean_ls).mean().item()
#     karlen_norm_std = np.array(std_ls).mean().item()
        
# karlen_datamodule = SequenceGeneDataModule(curve_dict_path=os.path.join(old_data_dir, 'karlen_curve_dict.pkl'),
#                                          target_df_path=os.path.join(old_data_dir, 'karlen_target_data.csv'),
#                                          norm_mean=karlen_norm_mean,
#                                          norm_std=karlen_norm_std,
#                                          num_workers=32)
# karlenloader = karlen_datamodule.test_dataloader()

# known_datamodule = SequenceGeneDataModule(curve_dict_path=os.path.join(old_data_dir, 'known_curve_dict.pkl'),
#                              target_df_path=os.path.join(old_data_dir, 'known_target_data.csv'),
#                              norm_mean=test_datamodule.norm_mean,
#                              norm_std=test_datamodule.norm_std,
#                              num_workers=32)
# knownloader = known_datamodule.test_dataloader()


WE ARE USING THE IMAGE SEQUENCE GENE DATASET


100%|██████████| 193558/193558 [00:03<00:00, 50041.66it/s]


In [31]:


seq_model = SeqModel().load_from_checkpoint(os.path.join(new_data_dir,'model_checkpoints/Seq_Model_large.ckpt'))
seq_model.to(device)
seq_model.eval()

### test
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(testloader):
    seq, gene, labels, ids = batch

    seq = seq.to(device)
    out = seq_model(seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_test_pred_df.to_csv(os.path.join(new_data_dir,'model_outputs/experimental/seq_test_pred_df.csv'), index=False)


### retest
# probs, groundtruths, curve_ids = [], [], []
# for batch in tqdm(retestloader):
#     seq, gene, labels, ids = batch

#     seq = seq.to(device)
#     out = seq_model(seq, gene)

#     probs.append(out[:,0].detach().cpu().numpy())
#     groundtruths.append(labels)
#     curve_ids.append(ids)
    
# seq_retest_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
#                                  'groundtruth_label': np.concatenate(groundtruths),
#                              'outputs': np.concatenate(probs).squeeze()})
# seq_retest_pred_df.to_csv(os.path.join(new_data_dir,'/model_outputs/experimental/seq_retest_pred_df.csv'), index=False)


# ### chip60
# probs, groundtruths, curve_ids = [], [], []
# for batch in tqdm(chip60loader):
#     seq, gene, labels, ids = batch

#     seq = seq.to(device)
#     out = seq_model(seq, gene)

#     probs.append(out[:,0].detach().cpu().numpy())
#     groundtruths.append(labels)
#     curve_ids.append(ids)
    
# seq_chip60_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
#                                  'groundtruth_label': np.concatenate(groundtruths),
#                              'outputs': np.concatenate(probs).squeeze()})
# seq_chip60_pred_df.to_csv(os.path.join(new_data_dir,'/model_outputs/experimental/seq_chip60_pred_df.csv'), index=False)



# ### karlen
# probs, groundtruths, curve_ids = [], [], []
# for batch in tqdm(karlenloader):
#     seq, gene, labels, ids = batch

#     seq = seq.to(device)
#     out = seq_model(seq, gene)

#     probs.append(out[:,0].detach().cpu().numpy())
#     groundtruths.append(labels)
#     curve_ids.append(ids)
    
# seq_karlen_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
#                                  'groundtruth_label': np.concatenate(groundtruths),
#                              'outputs': np.concatenate(probs).squeeze()})
# seq_karlen_pred_df.to_csv(os.path.join(new_data_dir,'/model_outputs/experimental/seq_karlen_pred_df.csv'), index=False)



# ### known
# probs, groundtruths, curve_ids = [], [], []
# for batch in tqdm(knownloader):
#     seq, gene, labels, ids = batch

#     seq = seq.to(device)
#     out = seq_model(seq, gene)

#     probs.append(out[:,0].detach().cpu().numpy())
#     groundtruths.append(labels)
#     curve_ids.append(ids)
    
# seq_known_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
#                                  'groundtruth_label': np.concatenate(groundtruths),
#                              'outputs': np.concatenate(probs).squeeze()})
# seq_known_pred_df.to_csv(os.path.join(new_data_dir,'/model_outputs/experimental/seq_known_pred_df.csv'), index=False)


/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/lightning/pytorch/utilities/migration/utils.py:55: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.1.1, which is newer than your current Lightning version: v2.0.9.post0
  rank_zero_warn(
  0%|          | 0/756 [00:01<?, ?it/s]


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 308, in _worker_loop
    data = fetcher.fetch(index)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torch/utils/data/_utils/collate.py", line 265, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torch/utils/data/_utils/collate.py", line 142, in collate
    return [collate(samples, collate_fn_map=collate_fn_map) for samples in transposed]  # Backw